In [16]:
#from transformers import AutoTokenizer
import json
import os
import random
random.seed(42)

In [17]:
#tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2-xl") #BPE tokenizer
#tokenizer = AutoTokenizer.from_pretrained("bert-base-cased") #WordPiece tokenizer
#tokenizer = AutoTokenizer.from_pretrained("xlnet-base-cased") #Unigram tokenizer

#t_label = "BPE"
#t_label = "WordPiece"
#t_label = "Unigram"

#real_path = "./distribution/part_"
#tokenized_path = "./distribution/tokenized/part_"

#new_path = "./distribution_same_topic_cyber_words"
new_path = "./CyberSecOnly"

def walk_directory(directory):
    l = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if not file.endswith(".csv"):
                l.append(os.path.join(root, file))
    return l

def create_article_list(file):
    #print(f"Creating article list from {file}")
    articles = []
        # Load the JSON file
    with open(file, "r", encoding="utf-8") as fl:
        f = json.load(fl)
        articles.append(f["original_doc"])
        articles.append(f['articles']['gpt4o']['article'])
        articles.append(f['articles']['claude3.5sonnet']['article'])
        articles.append(f['articles']['llama3.1-405b']['article'])
        articles.append(f['articles']['qwen1.5-110b']['article'])
        articles.append(f['articles']['watermarked']['article'])
    #print(f"Found {len(articles)} articles in {file}")
    return articles

def get_topic(path):
    topic =''
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
        topic = data["topic"]
    return topic

In [8]:
tmp = "Hello World! This is a test."
#print(tokenizer.tokenize(tmp))

In [18]:
import string

def tokenize_article(path):
    word_distribution = {}
    topic = get_topic(path)
    for article in create_article_list(path):
        # Preprocess the article: strip, lowercase, remove punctuation
        article = article.strip().lower()
        article = article.translate(str.maketrans('', '', string.punctuation))
        # Split into words (simple whitespace split)
        words = article.split()
        for word in words:
            if word not in word_distribution:
                word_distribution[word] = {"count": 1, "topic": [topic]}
            else:
                word_distribution[word]["count"] += 1
                if topic not in word_distribution[word]["topic"]:
                    word_distribution[word]["topic"].append(topic)
    return word_distribution


def tokenize_txt(path):
    word_distribution = {}
    #topic = get_topic(path)
    with open(path, "r", encoding="utf-8") as f:
        article = f.read()
        # Preprocess the article: strip, lowercase, remove punctuation
        article = article.strip().lower()
        article = article.translate(str.maketrans('', '', string.punctuation))
        # Split into words (simple whitespace split)
        words = article.split()
        for word in words:
            if word not in word_distribution:
                word_distribution[word] = {"count": 1} #, "topic": [topic]}
            else:
                word_distribution[word]["count"] += 1
                '''if topic not in word_distribution[word]["topic"]:
                    word_distribution[word]["topic"].append(topic)'''
    return word_distribution

In [19]:
def create_distribution(piecelist):
    final_word_distribution = {}
    for file in piecelist:
        print(f"Processing file: {file}")
        #word_distribution = tokenize_article(file)
        word_distribution = tokenize_txt(file)
        # Merge the token distributions
        for item in word_distribution:
            if item not in final_word_distribution:
                final_word_distribution[item] = word_distribution[item]
            else:
                final_word_distribution[item]["count"] += word_distribution[item]["count"]
                '''for topic in word_distribution[item]["topic"]:
                    if topic not in final_word_distribution[item]["topic"]:
                        final_word_distribution[item]["topic"].append(topic)'''
    sorted_distribution = sorted(final_word_distribution.items(), key=lambda x: x[1]["count"], reverse=True)
    output_file = f"./json_files/new_tokenizer_tests/RESULTS_cyber_new_WORD_distribution_CLEAN.json"
    os.remove(output_file) if os.path.exists(output_file) else None
    # Save the final token distribution to a JSON file
    with open(output_file, "w") as f:
        json.dump(sorted_distribution, f, indent=4)

In [20]:
'''for i in range(0,10):
    piece = walk_directory(f"{real_path}{i}")
    token_distribution = create_distribution(piece)'''

piece = walk_directory(new_path)
token_distribution = create_distribution(piece)

Processing file: ./CyberSecOnly\result_together_A_0.txt
Processing file: ./CyberSecOnly\result_together_A_1.txt
Processing file: ./CyberSecOnly\result_together_A_2.txt
Processing file: ./CyberSecOnly\result_together_A_3.txt
Processing file: ./CyberSecOnly\result_together_A_4.txt
Processing file: ./CyberSecOnly\result_together_A_5.txt
Processing file: ./CyberSecOnly\result_together_A_6.txt
Processing file: ./CyberSecOnly\result_together_A_7.txt
Processing file: ./CyberSecOnly\result_together_A_8.txt
Processing file: ./CyberSecOnly\result_together_A_9.txt
Processing file: ./CyberSecOnly\result_together_B_0.txt
Processing file: ./CyberSecOnly\result_together_B_1.txt
Processing file: ./CyberSecOnly\result_together_B_2.txt
Processing file: ./CyberSecOnly\result_together_B_3.txt
Processing file: ./CyberSecOnly\result_together_B_4.txt
Processing file: ./CyberSecOnly\result_together_B_5.txt
Processing file: ./CyberSecOnly\result_together_B_6.txt
Processing file: ./CyberSecOnly\result_together_

In [ ]:
'''# create a json file that contains all tokens that appear at most 2 times in two or more tokenizers
# open the three json files with the token distributions and load them
with open(f"./json_files/new_tokenizer_tests/w_cyber_new_token_distribution_BPE.json", "r") as f:
    bpe_distribution = json.load(f)
with open(f"./json_files/new_tokenizer_tests/w_cyber_new_token_distribution_WordPiece.json", "r") as f:
    wp_distribution = json.load(f)
with open(f"./json_files/new_tokenizer_tests/w_cyber_new_token_distribution_Unigram.json", "r") as f:
    unigram_distribution = json.load(f)

# create a new json file with the tokens that appear at most 2 times between the three distributions
common_tokens = {}
for token, data in bpe_distribution:
    if data["count"] <= 2:
        if token not in common_tokens:
            common_tokens[token] = {"count": data["count"], "topic": data["topic"], "tokenizer": "BPE"}
        else:
            common_tokens[token]["count"] += data["count"]
            for topic in data["topic"]:
                if topic not in common_tokens[token]["topic"]:
                    common_tokens[token]["topic"].append(topic)
            if "BPE" not in common_tokens[token]["tokenizer"]:
                common_tokens[token]["tokenizer"] += ", BPE"
for token, data in wp_distribution:
    if data["count"] <= 2:
        if token not in common_tokens:
            common_tokens[token] = {"count": data["count"], "topic": data["topic"], "tokenizer": "WordPiece"}
        else:
            common_tokens[token]["count"] += data["count"]
            for topic in data["topic"]:
                if topic not in common_tokens[token]["topic"]:
                    common_tokens[token]["topic"].append(topic)
            if "WordPiece" not in common_tokens[token]["tokenizer"]:
                common_tokens[token]["tokenizer"] += ", WordPiece"
for token, data in unigram_distribution:
    if data["count"] <= 2:
        if token not in common_tokens:
            common_tokens[token] = {"count": data["count"], "topic": data["topic"], "tokenizer": "Unigram"}
        else:
            common_tokens[token]["count"] += data["count"]
            for topic in data["topic"]:
                if topic not in common_tokens[token]["topic"]:
                    common_tokens[token]["topic"].append(topic)
            if "Unigram" not in common_tokens[token]["tokenizer"]:
                common_tokens[token]["tokenizer"] += ", Unigram"
# Save the common tokens to a JSON file
output_common_file = "./json_files/new_tokenizer_tests/w_cyber_common_tokens.json"
os.remove(output_common_file) if os.path.exists(output_common_file) else None   
with open(output_common_file, "w") as f:
    json.dump(common_tokens, f, indent=4)
    print(f"Common tokens saved to {output_common_file}")'''

'# create a jsoin file that contains all tokens that appear at most 2 times in two or more tokenizers\n# open the three json files with the token distributions and load them\nwith open(f"./json_files/new_tokenizer_tests/w_cyber_new_token_distribution_BPE.json", "r") as f:\n    bpe_distribution = json.load(f)\nwith open(f"./json_files/new_tokenizer_tests/w_cyber_new_token_distribution_WordPiece.json", "r") as f:\n    wp_distribution = json.load(f)\nwith open(f"./json_files/new_tokenizer_tests/w_cyber_new_token_distribution_Unigram.json", "r") as f:\n    unigram_distribution = json.load(f)\n\n# create a new json file with the tokens that appear at most 2 times between the three distributions\ncommon_tokens = {}\nfor token, data in bpe_distribution:\n    if data["count"] <= 2:\n        if token not in common_tokens:\n            common_tokens[token] = {"count": data["count"], "topic": data["topic"], "tokenizer": "BPE"}\n        else:\n            common_tokens[token]["count"] += data["cou

In [13]:
'''prefix = "w_"
final_token_distribution = {}

with open(f"./json_files/new_tokenizer_tests/{prefix}new_token_distribution_BPE.json", "r") as f:
    bpe_distribution = json.load(f)
with open(f"./json_files/new_tokenizer_tests/{prefix}new_token_distribution_WordPiece.json", "r") as f:
    wp_distribution = json.load(f)
with open(f"./json_files/new_tokenizer_tests/{prefix}new_token_distribution_Unigram.json", "r") as f:
    unigram_distribution = json.load(f)

with open(f"./json_files/new_tokenizer_tests/new_token_distribution_BPE.json", "r") as f:
    old_bpe_distribution = json.load(f)
with open(f"./json_files/new_tokenizer_tests/new_token_distribution_WordPiece.json", "r") as f:
    old_wp_distribution = json.load(f)
with open(f"./json_files/new_tokenizer_tests/new_token_distribution_Unigram.json", "r") as f:
    old_unigram_distribution = json.load(f)

with open(f"./json_files/topic_list/selected_words.json") as f:
    selected_words = json.load(f)

with open(f"./json_files/new_tokenizer_tests/{prefix}new_token_distribution.csv", "w") as f:
    f.write(f"Token,Clean_BPE,Clean_WordPiece,Clean_Unigram,Clean_Total,BPE,WordPiece,Unigram,Total\n")

print(selected_words)
# Iterate over the selected words and write their counts from each distribution
print(bpe_distribution)

for k,v in selected_words.items():
    current_tokens = {}
    bpe_count = 0
    wp_count = 0
    unigram_count = 0
    old_bpe_count = 0
    old_wp_count = 0
    old_unigram_count = 0
    with open(f"./json_files/new_tokenizer_tests/{prefix}new_token_distribution.csv", "a") as f:
        f.write(f"{k},,,,,,,,\n")
    for token in v:
        #count the number of times the token appears in each distribution
        for bpe_token, bpe_data in bpe_distribution:
            if bpe_token == token:
                bpe_count += bpe_data["count"]
        for wp_token, wp_data in wp_distribution:
            if wp_token == token:
                wp_count += wp_data["count"]
        for unigram_token, unigram_data in unigram_distribution:
            if unigram_token == token:
                unigram_count += unigram_data["count"]
        for old_bpe_token, old_bpe_data in old_bpe_distribution:
            if old_bpe_token == token:
                old_bpe_count += old_bpe_data["count"]
        for old_wp_token, old_wp_data in old_wp_distribution:
            if old_wp_token == token:
                old_wp_count += old_wp_data["count"]
        for old_unigram_token, old_unigram_data in old_unigram_distribution:
            if old_unigram_token == token:
                old_unigram_count += old_unigram_data["count"]

        with open(f"./json_files/new_tokenizer_tests/{prefix}new_token_distribution.csv", "a") as f:
            f.write(f"{token},{old_bpe_count},{old_wp_count},{old_unigram_count},{old_unigram_count+old_bpe_count+old_wp_count},{bpe_count},{wp_count},{unigram_count},{bpe_count + wp_count + unigram_count}\n")
            '''

'prefix = "w_"\nfinal_token_distribution = {}\n\nwith open(f"./json_files/new_tokenizer_tests/{prefix}new_token_distribution_BPE.json", "r") as f:\n    bpe_distribution = json.load(f)\nwith open(f"./json_files/new_tokenizer_tests/{prefix}new_token_distribution_WordPiece.json", "r") as f:\n    wp_distribution = json.load(f)\nwith open(f"./json_files/new_tokenizer_tests/{prefix}new_token_distribution_Unigram.json", "r") as f:\n    unigram_distribution = json.load(f)\n\nwith open(f"./json_files/new_tokenizer_tests/new_token_distribution_BPE.json", "r") as f:\n    old_bpe_distribution = json.load(f)\nwith open(f"./json_files/new_tokenizer_tests/new_token_distribution_WordPiece.json", "r") as f:\n    old_wp_distribution = json.load(f)\nwith open(f"./json_files/new_tokenizer_tests/new_token_distribution_Unigram.json", "r") as f:\n    old_unigram_distribution = json.load(f)\n\nwith open(f"./json_files/topic_list/selected_words.json") as f:\n    selected_words = json.load(f)\n\nwith open(f"./j

In [14]:
'''#count the topics in the common tokens and compare them to all the topics from the , token in one distribution
topics_count = {}
for token, data in common_tokens.items():
    for topic in data["topic"]:
        if topic not in topics_count:
            topics_count[topic] = 1
        else:
            topics_count[topic] += 1
print("Topics count in common tokens:")
for topic, count in topics_count.items():
    print(f"{topic}: {count}")

print("\n\n")
#print the number of common tokens
print(f"Number of common tokens: {len(common_tokens)}")
#sum all the counts of the topics to get the average number of topics per token
total_topics = sum(len(data["topic"]) for data in common_tokens.values())
average_topics = total_topics / len(common_tokens) if common_tokens else 0
print(f"Average number of topics per token: {average_topics:.2f}")'''

'#count the topics in the common tokens and compare them to all the topics from the , token in one distribution\ntopics_count = {}\nfor token, data in common_tokens.items():\n    for topic in data["topic"]:\n        if topic not in topics_count:\n            topics_count[topic] = 1\n        else:\n            topics_count[topic] += 1\nprint("Topics count in common tokens:")\nfor topic, count in topics_count.items():\n    print(f"{topic}: {count}")\n\nprint("\n\n")\n#print the number of common tokens\nprint(f"Number of common tokens: {len(common_tokens)}")\n#sum all the counts of the topics to get the average number of topics per token\ntotal_topics = sum(len(data["topic"]) for data in common_tokens.values())\naverage_topics = total_topics / len(common_tokens) if common_tokens else 0\nprint(f"Average number of topics per token: {average_topics:.2f}")'

In [15]:
'''#choose a random article from all in the distribution folder
def choose_random_article():
    real_path = "./distribution/real/"
    all_files = walk_directory(real_path)
    random_file = random.choice(all_files)
    return random_file

#select a random article and extract its topic. Based on the topic, select a list of 10 randomm tokens from the common tokens that are related to the topic
def select_random_tokens_for_topic(topic, common_tokens, num_tokens=10):
    related_tokens = [token for token, data in common_tokens.items() if topic in data["topic"]]
    if len(related_tokens) < num_tokens:
        print(f"Not enough related tokens for topic '{topic}'. Found: {len(related_tokens)}")
        return related_tokens
    return random.sample(related_tokens, num_tokens)

def main():
    random_article = choose_random_article()
    topic = get_topic(random_article)
    print(f"Selected article path: {random_article}")
    print(f"Randomly selected article topic: {topic}")
    
    common_tokens_file = "./json_files/new_tokenizer_tests/common_tokens.json"
    with open(common_tokens_file, "r") as f:
        common_tokens = json.load(f)
    selected_tokens = select_random_tokens_for_topic(topic, common_tokens)
    print(f"Selected tokens related to topic '{topic}': {selected_tokens}")
    #save the key facts and the additional facts from the article in two varibles
    with open(random_article, "r") as f:
        article_data = json.load(f)
        key_facts = article_data.get("key_facts", [])
        other_facts = article_data.get("other_facts", [])
        #choose 5 random other facts
        if len(other_facts) > 5:
            other_facts = random.sample(other_facts, 5)
        else:
            print(f"Not enough other facts. Found: {len(other_facts)}")
    print(f"Key facts: {key_facts}")
    print(f"Other facts: {other_facts}")

    kf_prompt = ", ".join(key_facts)
    of_prompt = ", ".join(other_facts)
    st_prompt = ", ".join(selected_tokens)

    prompt = f"""You are an AI assistant tasked with generating an article based on the following key facts (included in <keyfacts></keyfacts> tags) and additional facts (included in <additionalfacts></additionalfacts> tags). Use the provided tokens (included in <tokens></tokens> tags) as much as you can to enhance the article's content and ensure it is relevant to the topic '{topic}'.
Key Facts: <keyfacts>{kf_prompt}</keyfacts>
Additional Facts: <additionalfacts>{of_prompt}</additionalfacts>
Tokens to use: <tokens>{st_prompt}</tokens>
Please generate a coherent and informative article that incorporates these elements."""
    return prompt

if __name__ == "__main__":
    prompt = main()
    print("\nGenerated Prompt:")
    print(prompt)
    with open("./json_files/new_tokenizer_tests/generated_prompt.txt", "w") as f:
        f.write(prompt)'''

'#choose a random article from all in the distribution folder\ndef choose_random_article():\n    real_path = "./distribution/real/"\n    all_files = walk_directory(real_path)\n    random_file = random.choice(all_files)\n    return random_file\n\n#select a random article and extract its topic. Based on the topic, select a list of 10 randomm tokens from the common tokens that are related to the topic\ndef select_random_tokens_for_topic(topic, common_tokens, num_tokens=10):\n    related_tokens = [token for token, data in common_tokens.items() if topic in data["topic"]]\n    if len(related_tokens) < num_tokens:\n        print(f"Not enough related tokens for topic \'{topic}\'. Found: {len(related_tokens)}")\n        return related_tokens\n    return random.sample(related_tokens, num_tokens)\n\ndef main():\n    random_article = choose_random_article()\n    topic = get_topic(random_article)\n    print(f"Selected article path: {random_article}")\n    print(f"Randomly selected article topic: {t

In [16]:
'''w_common_tokens = "./json_files/new_tokenizer_tests/w_common_tokens.json"
common_tokens = "./json_files/new_tokenizer_tests/common_tokens.json"
selected_words = "./json_files/topic_list/selected_words.json"

with open(w_common_tokens, "r") as f:
    w_common_tokens_data = json.load(f)
with open(common_tokens, "r") as f:
    common_tokens_data = json.load(f)
with open(selected_words, "r") as f:
    selected_words_data = json.load(f)

for k,v in selected_words_data.items():
    for token in v:
        #count how many times the token appears in the w_common_tokens_data and common_tokens_data
        w_count = w_common_tokens_data.get(token, {}).get("count", 0)
        c_count = common_tokens_data.get(token, {}).get("count", 0)
        print(f"Token: {token}, W_Common Count: {w_count}, Common Count: {c_count}")'''

'w_common_tokens = "./json_files/new_tokenizer_tests/w_common_tokens.json"\ncommon_tokens = "./json_files/new_tokenizer_tests/common_tokens.json"\nselected_words = "./json_files/topic_list/selected_words.json"\n\nwith open(w_common_tokens, "r") as f:\n    w_common_tokens_data = json.load(f)\nwith open(common_tokens, "r") as f:\n    common_tokens_data = json.load(f)\nwith open(selected_words, "r") as f:\n    selected_words_data = json.load(f)\n\nfor k,v in selected_words_data.items():\n    for token in v:\n        #count how many times the token appears in the w_common_tokens_data and common_tokens_data\n        w_count = w_common_tokens_data.get(token, {}).get("count", 0)\n        c_count = common_tokens_data.get(token, {}).get("count", 0)\n        print(f"Token: {token}, W_Common Count: {w_count}, Common Count: {c_count}")'

In [1]:
#find all jsons that have the same topic and copy them to a new folder
import json
import os
import random
def copy_jsons_with_same_topic(source_dir, target_dir, topic):
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)
    for root, dirs, files in os.walk(source_dir):
        for file in files:
            if file.endswith(".json"):
                file_path = os.path.join(root, file)
                with open(file_path, "r", encoding="utf-8") as f:
                    data = json.load(f)
                    if data.get("topic") == topic:
                        target_path = os.path.join(target_dir, file)
                        with open(target_path, "w", encoding="utf-8") as target_file:
                            json.dump(data, target_file, indent=4)
                        print(f"Copied {file} to {target_dir}")
source_directory = "./distribution_clean"
target_directory = "./distribution_divided"
with open("./json_files/topic_list/selected_words.json", "r") as f:
    selected_topics = json.load(f)
    list_of_topics = list(selected_topics.keys())

for topic_to_copy in list_of_topics:
    string_path = '/' + topic_to_copy.replace(" ", "_").replace("/", "_").lower()
    copy_jsons_with_same_topic(source_directory, target_directory + string_path, topic_to_copy)
    print(f"All JSON files with topic '{topic_to_copy}' have been copied to {target_directory}.")

Copied 2960.json to ./distribution_divided/company_policies
Copied 2970.json to ./distribution_divided/company_policies
Copied 3000.json to ./distribution_divided/company_policies
Copied 3010.json to ./distribution_divided/company_policies
Copied 3180.json to ./distribution_divided/company_policies
Copied 3390.json to ./distribution_divided/company_policies
Copied 3430.json to ./distribution_divided/company_policies
Copied 3550.json to ./distribution_divided/company_policies
Copied 3580.json to ./distribution_divided/company_policies
Copied 3590.json to ./distribution_divided/company_policies
Copied 0061.json to ./distribution_divided/company_policies
Copied 2121.json to ./distribution_divided/company_policies
Copied 2941.json to ./distribution_divided/company_policies
Copied 2971.json to ./distribution_divided/company_policies
Copied 3021.json to ./distribution_divided/company_policies
Copied 3051.json to ./distribution_divided/company_policies
Copied 3081.json to ./distribution_divid

In [18]:
'''import glob

def remove_watermarked_from_jsons(path):
    """
    Remove all 'watermarked' elements from JSON files in the given directory (recursively).
    Modifies files in place.
    """
    json_files = glob.glob(os.path.join(path, '**', '*.json'), recursive=True)
    for file in json_files:
        with open(file, 'r', encoding='utf-8') as f:
            try:
                data = json.load(f)
            except Exception as e:
                print(f"Error reading {file}: {e}")
                continue
        changed = False
        # Remove 'watermarked' key if present at the top level
        if 'watermarked' in data:
            del data['watermarked']
            changed = True
        # Remove 'watermarked' from nested 'articles' if present
        if 'articles' in data and isinstance(data['articles'], dict):
            if 'watermarked' in data['articles']:
                del data['articles']['watermarked']
                changed = True
        if changed:
            with open(file, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=4, ensure_ascii=False)
                print(f"Updated: {file}")


remove_watermarked_from_jsons("./distribution_same_topic_cyber_words")'''

Updated: ./distribution_same_topic_cyber_words\0050.json
Updated: ./distribution_same_topic_cyber_words\0060.json
Updated: ./distribution_same_topic_cyber_words\0079.json
Updated: ./distribution_same_topic_cyber_words\0104.json
Updated: ./distribution_same_topic_cyber_words\0120.json
Updated: ./distribution_same_topic_cyber_words\0123.json
Updated: ./distribution_same_topic_cyber_words\0141.json
Updated: ./distribution_same_topic_cyber_words\0194.json
Updated: ./distribution_same_topic_cyber_words\0205.json
Updated: ./distribution_same_topic_cyber_words\0212.json
Updated: ./distribution_same_topic_cyber_words\0221.json
Updated: ./distribution_same_topic_cyber_words\0237.json
Updated: ./distribution_same_topic_cyber_words\0276.json
Updated: ./distribution_same_topic_cyber_words\0292.json
Updated: ./distribution_same_topic_cyber_words\0293.json
Updated: ./distribution_same_topic_cyber_words\0352.json
Updated: ./distribution_same_topic_cyber_words\0390.json
Updated: ./distribution_same_to